# [M3] XÂY MODEL AI SÂU HƠN BẰNG PYTORCH
## Buổi S02: PyTorch Data Pipeline: Tùy Biến Dataset & DataLoader

---

### Metadata Khóa Học
* **Module:** M3 — XÂY MODEL AI SÂU HƠN BẰNG PYTORCH
* **Buổi:** S02
* **Thời lượng dự kiến:** 120 phút
* **Sản phẩm đầu ra (Session Output):** Notebook xây dựng Custom Dataset và DataLoader hoàn chỉnh
* **Mức độ liên quan VOAI / Olympic AI:** Quản lý bộ nhớ tối ưu, loại bỏ triệt để nghẽn cổ chai nạp dữ liệu khi thi đấu.

#### Mục tiêu học tập (Learning Objectives)
- Hiểu lý do tại sao không thể nạp toàn bộ tập dữ liệu khổng lồ vào RAM/VRAM cùng một lúc (Out of Memory - OOM).
- Kế thừa và tùy biến lớp dữ liệu `torch.utils.data.Dataset` thông qua 3 phương thức chuẩn mực: `__init__`, `__len__`, `__getitem__`.
- Sử dụng `torch.utils.data.DataLoader` để tự động hóa việc chia Mini-batch, xáo trộn (Shuffle) và nạp dữ liệu đa tiến trình.
- Tích hợp tiền xử lý và chuyển đổi kiểu dữ liệu Tensor trực tiếp trong luồng nạp dữ liệu.

#### Điều kiện tiên quyết (Prerequisites)
- Hoàn thành Buổi M3_S01 (PyTorch Tensors).
- Hiểu cú pháp Lập trình hướng đối tượng (OOP) cơ bản trong Python (kế thừa class).

---


## 1. WHY? — TẠI SAO PHẢI HỌC CHỦ ĐỀ NÀY?


Trong các bài thi Olympic AI hoặc các dự án thực tế, tập dữ liệu có thể chứa hàng triệu dòng dữ liệu bảng hoặc hàng trăm nghìn tấm ảnh độ phân giải cao (nhiều Gigabytes). Nếu bạn cố gắng ép toàn bộ tập dữ liệu thành một Tensor khổng lồ rồi đưa vào huấn luyện, card đồ họa GPU hoặc RAM máy tính sẽ lập tức nổ tung với thông báo lỗi kinh hoàng: `CUDA out of memory`!

PyTorch giải quyết triệt để vấn đề này bằng một cặp đôi hoàn hảo: **`Dataset`** (đóng vai trò như một thủ kho biết chính xác từng món hàng nằm ở đâu) và **`DataLoader`** (đóng vai trò như đội ngũ xe tải chuyên chở từng chuyến hàng nhỏ - Mini-batch - nạp vào GPU đều đặn theo thời gian thực).



## 2. CONCEPT — NỀN TẢNG LÝ THUYẾT & TRỰC GIÁC


### 1. Kiến trúc tùy biến `torch.utils.data.Dataset`
Để tạo một Dataset tùy biến trong PyTorch, bạn bắt buộc phải kế thừa lớp `Dataset` và ghi đè (override) đúng 3 phương thức:
1. `__init__(self, ...)`: Nạp dữ liệu thô (từ DataFrame, file CSV, hoặc mảng NumPy) và lưu trữ các biến cấu hình.
2. `__len__(self)`: Trả về tổng số lượng mẫu trong tập dữ liệu (`len(self.data)`).
3. `__getitem__(self, idx)`: Trả về đúng một cặp mẫu thử $(X_{idx}, y_{idx})$ tại chỉ số `idx` cụ thể, đã được chuyển hóa sang kiểu `torch.Tensor`.

### 2. Các tham số cốt tử của `DataLoader`
* `batch_size`: Số lượng mẫu dữ liệu được gom lại trong một lần chuyển giao cho mô hình (ví dụ: 32, 64, 128).
* `shuffle`: Có xáo trộn ngẫu nhiên thứ tự các mẫu ở đầu mỗi epoch không? (Luôn đặt `True` cho tập Train để phá vỡ tương quan thứ tự, và đặt `False` cho tập Validation/Test để giữ nguyên thứ tự đánh giá).
* `num_workers`: Số lượng tiến trình con chạy song song dưới nền để đọc dữ liệu từ đĩa cứng (giúp GPU không bị đói dữ liệu).
* `drop_last`: Có bỏ qua batch cuối cùng nếu số lượng mẫu còn sót lại không đủ một full batch hay không.



## 3. VISUAL INTUITION — TRỰC QUAN HÓA KIẾN THỨC


```
Luồng dữ liệu PyTorch Data Pipeline:
[Toàn bộ File Dữ liệu / Đĩa cứng]
               │
               ▼
   [Custom Dataset Class] ──> Cung cấp __getitem__(idx) cho từng mẫu đơn lẻ
               │
               ▼
        [DataLoader]
      ├──> Gom nhóm mẫu thành Mini-Batches (batch_size=32)
      ├──> Xáo trộn ngẫu nhiên (shuffle=True)
      └──> Nạp song song đa luồng (num_workers=2)
               │
               ▼
[Vòng lặp Training Loop] ──> for batch_X, batch_y in train_loader: ...
```



## 4. CODE IMPLEMENTATION & 5. WALKTHROUGH

*Toàn bộ code dưới đây được thiết kế thực tế, có thể chạy độc lập không phụ thuộc tải dữ liệu bên ngoài.*

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np

# 1. Khoi tao mot tap du lieu bang mo phong ve du doan suc khoe
np.random.seed(42)
n_records = 200

raw_df = pd.DataFrame({
    'Heart_Rate': np.random.normal(75, 12, size=n_records).round(1),
    'Blood_Pressure': np.random.normal(120, 15, size=n_records).round(1),
    'Cholesterol': np.random.normal(200, 30, size=n_records).round(1),
    'Has_Heart_Risk': np.random.choice([0, 1], size=n_records, p=[0.7, 0.3])
})

# 2. Dinh nghia lop Custom Tabular Dataset ke thua tu torch.utils.data.Dataset
class HealthTabularDataset(Dataset):
    def __init__(self, dataframe, is_train=True):
        self.features = dataframe[['Heart_Rate', 'Blood_Pressure', 'Cholesterol']].values
        self.labels = dataframe['Has_Heart_Risk'].values
        self.is_train = is_train
        
    def __len__(self):
        return len(self.features)
        
    def __getitem__(self, idx):
        # Trich xuat mau tai chi so idx va ep kieu sang torch.float32
        x = torch.tensor(self.features[idx], dtype=torch.float32)
        y = torch.tensor(self.labels[idx], dtype=torch.float32) # Co the la float32 neu dung BCEWithLogitsLoss
        return x, y

# Tao doi tuong Dataset
health_dataset = HealthTabularDataset(raw_df)
print(f"Tong so mau trong Dataset: {len(health_dataset)}")

# Kiem tra thu mau dau tien
sample_x, sample_y = health_dataset[0]
print(f"Mau thu dau tien (Index 0) -> x: {sample_x}, y: {sample_y}")

### 🔍 Code Walkthrough (Khối 1)
Khối code minh họa việc tạo một lớp Custom Dataset chuẩn mực:
- Kế thừa `Dataset` từ `torch.utils.data`.
- Cài đặt đầy đủ 3 phương thức `__init__`, `__len__`, `__getitem__`.
- Đảm bảo dữ liệu trả về từ `__getitem__` luôn là các PyTorch Tensor với kiểu dữ liệu chuẩn (`float32`).


In [ ]:
# 3. Dong goi Dataset vao DataLoader va duyet qua cac Mini-Batches
batch_size = 32

train_loader = DataLoader(
    dataset=health_dataset,
    batch_size=batch_size,
    shuffle=True,       # Xao tron du lieu de mo hinh khong hoc theo thu tu
    drop_last=False     # Giu lai batch cuoi du co the nho hon 32
)

print(f"Tong so batch trong 1 Epoch: {len(train_loader)} (moi batch chua toi da {batch_size} mau)")

# Duyet thu qua Batch dau tien de kiem tra kich thuoc Tensor
for batch_idx, (batch_features, batch_labels) in enumerate(train_loader, start=1):
    print(f"Batch {batch_idx:2d} -> Features Shape: {batch_features.shape} | Labels Shape: {batch_labels.shape}")
    if batch_idx == 2:
        print("... Cac batch tiep theo nap tuong tu ...")
        break

### 🔍 Code Walkthrough (Khối 2)
Khối code khởi tạo và kiểm tra `DataLoader`:
- Tự động chia 200 mẫu thành 7 batch (6 batch 32 mẫu và 1 batch cuối 8 mẫu).
- Kích thước tensor đầu ra của mỗi batch là `(batch_size, num_features)` — sẵn sàng đưa trực tiếp vào tầng đầu tiên của mạng nơ-ron.


## 6. PRACTICE — BÀI TẬP THỰC HÀNH


### Bài 1 (Easy): Tích hợp Chuẩn hóa dữ liệu vào Dataset
Sửa lại class `HealthTabularDataset` để nhận thêm tham số `mean` và `std`. Trong `__getitem__`, áp dụng công thức chuẩn hóa $x_{norm} = \frac{x - mean}{std}$ trước khi trả về tensor.

### Bài 2 (Easy): Phân chia Train/Validation Dataset
Sử dụng hàm `torch.utils.data.random_split(health_dataset, [0.8, 0.2])` để chia dataset thành 2 tập: `train_set` (80%) và `val_set` (20%). Khởi tạo 2 DataLoader tương ứng (lưu ý: `train_loader` có `shuffle=True`, còn `val_loader` có `shuffle=False`).

### Bài 3 (Medium): Xử lý Batch không đều với `drop_last=True`
Chạy lại DataLoader với tham số `drop_last=True`. Quan sát xem tổng số batch trong 1 epoch thay đổi như thế nào và giải thích khi nào trong thi đấu Olympic ta nên bật `drop_last=True` (gợi ý: khi một số tầng BatchNorm yêu cầu batch size lớn hơn 1).

### Bài 4 (Medium): Kiểm tra tốc độ nạp dữ liệu với `num_workers`
Tìm hiểu tham số `num_workers` trong DataLoader. Thử nghiệm đặt `num_workers=0` (mặc định chạy trên tiến trình chính) và `num_workers=2` (đa tiến trình). Nêu lưu ý quan trọng khi dùng `num_workers > 0` trên hệ điều hành Windows (yêu cầu khối lệnh `if __name__ == '__main__':`).

### Bài 5 (Challenge): Viết Custom Collate Function
Trong nhiều bài toán xử lý văn bản hoặc đồ thị, các mẫu trong cùng một batch có thể có độ dài khác nhau. Hãy tìm hiểu tham số `collate_fn` của DataLoader và viết một hàm collate đơn giản tự động đệm thêm số 0 (padding) cho các vector có kích thước không đồng nhất.



## 7. VOAI CONNECTION — CHIẾN LƯỢC PHÒNG THI OLYMPIC


Trong các bài thi Deep Learning của Olympic AI, nghẽn cổ chai lớn nhất thường **không nằm ở GPU mà nằm ở CPU nạp dữ liệu (Data Loading Bottleneck)**! Nếu bạn viết hàm `__getitem__` quá nặng (ví dụ: mở file đĩa liên tục, tiền xử lý chuỗi phức tạp bằng vòng lặp Python bên trong getitem), GPU sẽ phải ngồi chơi xơi nước 80% thời gian chờ CPU chuyển dữ liệu sang. Thiết kế `Dataset` gọn gàng, tính toán trước các thao tác nặng trong `__init__`, và dùng `DataLoader` hợp lý sẽ giúp bạn rút ngắn thời gian huấn luyện từ 1 giờ xuống còn 10 phút!



## 8. DEBUGGING CHALLENGE — BẮT LỖI THỰC CHIẾN


**Tình huống thực tế:** Một thí sinh viết `__getitem__` trả về nhãn mục tiêu dạng số nguyên nhưng quên chuyển sang Tensor, hoặc chuyển sai kiểu dữ liệu thành `torch.float64` (Double) thay vì `torch.float32`:

```python
# CODE CÓ LỖI KIỂU DỮ LIỆU TENSOR KHI NẠP:
# PyTorch mac dinh cac lop Linear yeu cau dau vao la Float32!
# Neu tra ve Float64 (DoubleTensor), mo hinh se văng loi:
# RuntimeError: Expected object of scalar type Float but got scalar type Double for argument #4 'mat1'
```

**Nhiệm vụ của bạn:**
1. Giải thích tại sao PyTorch chuẩn hóa toàn bộ tính toán mạng nơ-ron ở kiểu dữ liệu `torch.float32` (Single Precision) thay vì `float64`.
2. Sửa lỗi triệt để trong `__getitem__` bằng cách luôn tường minh `dtype=torch.float32`.



## 9. KNOWLEDGE CHECK — KIỂM TRA ĐỘ HIỂU BẢN CHẤT


1. Khi kế thừa lớp `torch.utils.data.Dataset`, ba phương thức nào là BẮT BUỘC phải cài đặt?
   - A. `__start__`, `__run__`, `__stop__`
   - B. `__init__`, `__len__`, `__getitem__`
   - C. `fit`, `predict`, `score`
   - D. `forward`, `backward`, `step`

2. Tham số `shuffle=True` trong DataLoader có ý nghĩa gì và nên được áp dụng cho tập dữ liệu nào?
   - A. Xáo trộn thứ tự các cột đặc trưng; áp dụng cho cả Train và Test.
   - B. Xáo trộn ngẫu nhiên thứ tự các hàng mẫu thử ở đầu mỗi epoch; nên áp dụng cho tập Train để tránh thiên lệch thứ tự học.
   - C. Xóa các hàng ngẫu nhiên để giảm dung lượng RAM.
   - D. Chỉ áp dụng cho tập Test để tăng độ khó.

3. Phương thức `__len__(self)` trong Custom Dataset được gọi khi người dùng thực hiện câu lệnh nào?
   - A. `dataset[0]`
   - B. `len(dataset)`
   - C. `dataset.size()`
   - D. `dataset.shape`

4. Hiện tượng `CUDA out of memory` (OOM) trong quá trình huấn luyện mạng nơ-ron thường được khắc phục nhanh nhất bằng cách điều chỉnh tham số nào?
   - A. Tăng số lượng epochs.
   - B. Giảm giá trị `batch_size` trong DataLoader (ví dụ giảm từ 128 xuống 64 hoặc 32).
   - C. Tăng Learning Rate lên gấp 10 lần.
   - D. Tắt mạng Internet.

5. Tham số `drop_last=True` trong DataLoader thực hiện nhiệm vụ gì?
   - A. Xóa mẫu dữ liệu cuối cùng của tập dữ liệu vĩnh viễn.
   - B. Bỏ qua mini-batch cuối cùng nếu số lượng phần tử của nó nhỏ hơn `batch_size` quy định.
   - C. Xóa tầng cuối cùng của mạng nơ-ron.
   - D. Bỏ qua epoch cuối cùng.



## 10. SESSION SUMMARY — TỔNG KẾT BUỔI HỌC


* **What I learned:**
  - Nguyên nhân và cách phòng chống lỗi tràn bộ nhớ (OOM) bằng xử lý theo Batch.
  - Cấu trúc 3 phương thức chuẩn mực của `torch.utils.data.Dataset`.
  - Cơ chế hoạt động của `DataLoader`: Batching, Shuffling, Multi-processing.
* **What I can now do:**
  - Tự tay đóng gói bất kỳ bảng dữ liệu nào thành PyTorch Dataset chuyên nghiệp.
  - Thiết lập luồng nạp dữ liệu Mini-batch chuẩn hóa sẵn sàng cấp liệu cho mạng nơ-ron.
* **What I still need to practice:**
  - Thiết kế kiến trúc mạng nơ-ron hoàn chỉnh bằng **`torch.nn.Module`** ở Buổi 3.



## 11. NEXT SESSION — BƯỚC ĐI TIẾP THEO


Trong **Buổi M3_S03**, chúng ta sẽ chính thức xây dựng bộ não nơ-ron: **Thiết Kế Kiến Trúc Mạng Nơ-ron với `torch.nn.Module`** — Tìm hiểu cấu trúc Perceptron đa tầng (MLP), các hàm kích hoạt (ReLU, Sigmoid), và cách quản lý tham số mạng!



---

## 🎯 Syllabus Traceability (Đối chiếu Syllabus)

Buổi học này đóng góp trực tiếp vào mục tiêu của **M3**:
- Hiểu Dataset & DataLoader trong PyTorch ✓
- Tùy biến Custom Dataset (__init__, __len__, __getitem__) ✓
- Quản lý batch_size, shuffle & nạp dữ liệu an toàn bộ nhớ ✓

* **Session Deliverable:** Mã nguồn Data Pipeline tùy biến đóng gói dữ liệu bảng thành Mini-batches
* **Đảm bảo chuẩn Syllabus Lock:** Nội dung nằm đúng chuẩn 100%, không lan man, giải quyết trực tiếp năng lực cốt lõi.
